# Does uniform pseudo-labeling harm an identifiable node population in GLEM?

**Project summary — the idea, what was built, what was run, and what came back.**

Companion notebooks: [`glem_harm_results.ipynb`](glem_harm_results.ipynb) (the
preregistered measurement in detail), [`glem_results.ipynb`](glem_results.ipynb)
(per-step exploration), [`glem_gate_selection.ipynb`](glem_gate_selection.ipynb)
(choosing a deployable gate). Protocol and full amendment log:
[`EXPERIMENT.md`](../EXPERIMENT.md).

---

## The idea

GLEM (Zhao et al., ICLR 2023) alternates an **E-step** — train the LM on the GNN's
pseudo-labels — with an **M-step** — train the GNN on the LM's embeddings and
pseudo-labels. Each step weights its pseudo-label term by a single global scalar:
α for the LM step, β for the GNN step. That encodes an assumption:

> *the value of the teacher's signal does not depend on the node.*

The hypothesis was that this is false in a **predictable, directional** way. Outside
its inductive bias a model is not merely uncertain but **confidently wrong** — the
GNN on **low-homophily** nodes, the LM on **semantically ambiguous** text. So a
model teaching from inside that region should *corrupt* student nodes that were
previously correct, and the net effect there should be **negative**.

Two exogenous signals make that testable without circularity: local homophily and
kNN semantic ambiguity, neither of which depends on the model being evaluated.

## How the work actually unfolded

| phase | question | answer |
|---|---|---|
| **1. Instrument** | can before/after/teacher be recovered per step? | yes — but GLEM overwrites its own logits, so an archive had to be built first |
| **2. Harm measurement** | is NCS negative where the teacher is out of bias? | **no** — 14 not supported, 2 weakly supported, 10 no test |
| **3. Why not?** | what breaks the prediction? | GLEM's **asymmetric α/β** already down-weights the direction where the teacher is weak |
| **4. Mechanism** | does the student adopt the teacher's wrong labels? | **yes, 1.7–5.8×** — the mechanism is real, it just does not net out |
| **5. Ceiling** | would perfect gating pay? | **yes, +3.4 / +4.4pp** on arxiv; positive on 17 of 18 cells |
| **6. Confidence gate** | does the field's standard fix capture it? | **no, ≈0** — and we can say precisely why |
| **7. Signal gate** | do exogenous signals reach what confidence cannot? | they reach the population but **do not beat confidence** |
| **8. Feature channel** | is the harm in *how* the label is delivered? | **yes — Supported.** −1.21pp accuracy, concentrated out-of-bias |
| **9. Why** | is it the label, or the *unmasked* channel? | **the channel.** Matching train/inference reliability recovers 102% |
| **10. Unification** | is harm a property of the *pathway*? | **no — of exposure.** β=0.8 on the loss channel does *more* damage than the unscaled feature channel |
| **11. Second framework** | does it hold outside GLEM? | **directionally yes** — in GNN-as-Judge the optimum is *zero* transferred pairs (13pp range) |
| **12. Floor** | when does any of this apply? | at 20 labels/class the LM is 4pp above chance — the loop degenerates |

The pivot at phase 3 is the substance of the project: the original prediction was
wrong, and being wrong turned out to be more informative than being right, because the
reason is measurable. Phase 8 then found the prediction **was** right — in a channel §4
never measured. Phase 3's own first explanation ("the teacher stays better than the
student") was itself later withdrawn: it holds only for `gnn→lm`, and on arxiv by 0.9pp.
In `lm→gnn` the teacher is *worse* than the student in 12 of 12 cells, and NCS is still
non-negative in 5 of them.

In [10]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / 'src' / 'probe').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
ANALYSIS = ROOT / 'temp' / 'probe_analysis'
PROBE = ROOT / 'temp' / 'probe_output'
pd.set_option('display.width', 220)

ncs = pd.read_csv(ANALYSIS / 'ncs_long.csv')
verdicts = pd.read_csv(ANALYSIS / 'verdicts.csv')

runs = sorted(PROBE.glob('*/standard/*/seed*/steps.jsonl'))
inv = pd.DataFrame([{'dataset': r.parts[-5], 'arm': r.parts[-3],
                     'seed': int(r.parts[-2][4:])} for r in runs])
print(f'{len(inv)} completed runs | {inv.dataset.nunique()} dataset variants '
      f'| {inv.arm.nunique()} arms')
print()
print('runs per arm:')
print(inv.groupby('arm').size().sort_values(ascending=False).to_string())

367 completed runs | 13 dataset variants | 30 arms

runs per arm:
arm
published             39
alpha0_li_T           39
oracle_random         36
oracle                36
conf_gate80           15
beta_high_rand80      13
sig_gate80_lm         13
sig_gate80_gnn        12
published_li_F        12
alpha0_li_F           12
b30                   11
beta_high             11
b30_rand80            10
beta_high_sig80       10
b30_sig80lm           10
b30_sig80gnn           9
b05_rand80             9
b80_sig80gnn           9
conf_gate60            9
b30_conf80             8
b80_conf80             8
conf_gate90            6
published_li_T         6
teacher_consistent     6
sig_gate80             4
mask_train             3
alpha0_li_T_only       3
sig_gate90             3
unimp_mask             3
mask_pseudo            2


## Phase 1 — the instrument, and the bug that made it necessary

GLEM's teacher and student communicate **only** through fp16 memmaps on disk: the
GNN writes its predictions, the LM reads them as `pseudo_label_file`, and vice
versa. That is convenient — every before/after/teacher triple is already on disk, so
no extra forward passes are needed.

But for `em_iter >= 0` **the paths carry no iteration index**, so each M-step
clobbers the previous GNN prediction and each E-step the previous LM one. Before/after
pairs are destroyed as a run proceeds. Nothing downstream was measurable until that
was fixed.

`src/probe/` archives every write under an iteration-keyed path, plus the teacher
snapshot each step actually consumed, the node ids that actually received a
pseudo-label (unreconstructable afterwards — the GNN resamples every epoch), and the
split. Every hook is a no-op unless `GLEM_PROBE_DIR` is set, so an unprobed run is
byte-identical to stock GLEM.

**Fidelity check first**, because a broken reproduction would void everything after it.

In [11]:
fidelity = pd.DataFrame([
    {'metric': 'test accuracy', 'reproduced': 0.769973, 'paper': 0.7697, 'paper_sd': 0.0019},
    {'metric': 'val accuracy',  'reproduced': 0.775932, 'paper': 0.7749, 'paper_sd': 0.0017},
])
fidelity['delta'] = (fidelity.reproduced - fidelity.paper).round(5)
fidelity['within_paper_sd'] = fidelity.delta.abs() <= fidelity.paper_sd
print('GLEM + RevGAT on ogbn-arxiv, published hyperparameters, nothing tuned')
print(fidelity.to_string(index=False))

GLEM + RevGAT on ogbn-arxiv, published hyperparameters, nothing tuned
       metric  reproduced  paper  paper_sd   delta  within_paper_sd
test accuracy    0.769973 0.7697    0.0019 0.00027             True
 val accuracy    0.775932 0.7749    0.0017 0.00103             True


## Phase 2 — the preregistered measurement

`EXPERIMENT.md` fixed every bin, threshold, axis and decision rule **before any
measurement existed** (commit `33a3601`). Eighteen amendments record every deviation
since, with dates and reasons, including the ones that hurt. Twenty-eight amendments
now, and A26 keeps the running tally of registered predictions: **three right, four
wrong**.

The rule (§11), per (dataset, direction):

- **Supported** — NCS significantly negative out-of-bias, positive elsewhere, sign
  stable across seeds, **and** absent from the α=β=0 control.
- **Weakly supported** — NCS positive everywhere but significantly *lower*
  out-of-bias, with teacher accuracy degrading across the axis.
- **Not supported** — flat, **or** the pattern reproduces in the control.

In [12]:
show = verdicts[['dataset', 'direction', 'published_ncs_oob', 'published_ncs_in',
                 'published_ncs_gap', 'published_gap_sd', 'published_sign_stable',
                 'published_tacc_oob', 'published_tacc_in', 'verdict']].copy()
show['dataset'] = show.dataset.str.replace('_TAG', '', regex=False).str.replace('_TA', '', regex=False)
print(show.sort_values(['verdict', 'dataset']).round(4).to_string(index=False))
print()
print(verdicts.verdict.value_counts().to_string())

          dataset direction  published_ncs_oob  published_ncs_in  published_ncs_gap  published_gap_sd published_sign_stable  published_tacc_oob  published_tacc_in          verdict
         citeseer   gnn->lm                NaN               NaN                NaN               NaN                   NaN                 NaN                NaN          no test
             cora   gnn->lm                NaN               NaN                NaN               NaN                   NaN                 NaN                NaN          no test
          cornell   gnn->lm                NaN               NaN                NaN               NaN                   NaN                 NaN                NaN          no test
   cornell+revgat   gnn->lm                NaN               NaN                NaN               NaN                   NaN                 NaN                NaN          no test
           pubmed   gnn->lm                NaN               NaN                NaN               Na

### What the verdict table says

**One weakly-supported cell: arxiv `gnn->lm`.** NCS is positive in both bins
(+0.0170 in-bias, +0.0062 out-of-bias) but the gap is **−0.0108 ± 0.0017**,
sign-stable across three seeds, p = 6.2e−19, with teacher accuracy falling
**0.957 → 0.580** across the axis. The α=β=0 control cannot reproduce it: its
per-seed gaps are +0.0001 / +0.0103 / −0.0169, sd 0.0137 — eight times the published
spread, flipping sign twice.

**Everything else is null or untestable**, and the *reasons* matter more than the
counts:

- **A6 — degenerate bins.** cora, citeseer and pubmed have median local homophily
  **1.000** with ~65% of nodes tied there, so the median split leaves one bin empty.
  Not a null: a missing measurement.
- **A4 — no power.** WebKB's E-step sees 6–11 nodes per step under
  `lm_pl_ratio=0.1`. Fixed later by re-running those datasets on RevGAT (A13), which
  uses `lm_pl_ratio=1` and lifts the population to 71–101.
- **Sign instability.** Four cells had the *predicted* negative sign but flipped
  across seeds — §10 rejects those regardless of p-value.

The single positive result exists only because arxiv is the one dataset with both a
non-degenerate homophily axis **and** full pseudo-label coverage.

## Phase 3 — why the prediction failed

Two post-hoc analyses, both labelled as such and excluded from the verdict.

**The headroom confound (A8).** NCS divides by bin size, but corruption is capped by
how many nodes the student had right to begin with — and student accuracy is 6–19
points *lower* out-of-bias on every dataset. Conditioning on corruptible nodes, the
published arm does show elevated corruption out-of-bias (1.26–5.83×) — **but the
α=β=0 control reproduces it at equal or larger magnitude**. That fragility belongs
to low-margin nodes under any retraining, not to the pseudo-label term.

**Absolute vs relative teacher weakness — the actual explanation.** The hypothesis
assumed "teacher outside its inductive bias" implies "teacher worse than the
student". It does not.

In [13]:
t = ncs[(ncs.axis == 'teacher') & (ncs.bin_scheme == 'median')
        & (ncs.arm == 'published') & (~ncs.low_n_flag)]
OOB = {'gnn->lm': 'low', 'lm->gnn': 'high'}
e = (t.groupby(['direction', 'dataset', 'bin'])
     .agg(n=('n', 'sum'), teacher=('teacher_acc', 'mean'),
          student_before=('student_acc_before', 'mean')).reset_index())
e['role'] = np.where([b == OOB[d] for d, b in zip(e.direction, e['bin'])],
                     'OUT-OF-BIAS', 'in-bias')
e['teacher_edge'] = (e.teacher - e.student_before).round(4)
e['dataset'] = e.dataset.str.replace('_TAG', '', regex=False).str.replace('_TA', '', regex=False)
print('teacher_edge = teacher accuracy - student accuracy BEFORE the step')
print('negative => the teacher really is worse than the student on those nodes')
print(e[['direction', 'dataset', 'role', 'n', 'teacher', 'student_before',
         'teacher_edge']].sort_values(['direction', 'dataset', 'role'])
      .round(3).to_string(index=False))

teacher_edge = teacher accuracy - student accuracy BEFORE the step
negative => the teacher really is worse than the student on those nodes
direction           dataset        role      n  teacher  student_before  teacher_edge
  gnn->lm             arxiv OUT-OF-BIAS 235296    0.580           0.572         0.009
  gnn->lm             arxiv     in-bias 235116    0.957           0.918         0.039
  gnn->lm    cornell+revgat OUT-OF-BIAS    322    0.768           0.400         0.368
  gnn->lm      texas+revgat OUT-OF-BIAS    334    0.832           0.500         0.332
  gnn->lm washington+revgat OUT-OF-BIAS    294    0.783           0.643         0.140
  gnn->lm washington+revgat     in-bias    229    0.776           0.607         0.169
  gnn->lm            wikics OUT-OF-BIAS   1185    0.685           0.499         0.185
  gnn->lm            wikics     in-bias   1161    0.966           0.744         0.222
  gnn->lm  wisconsin+revgat OUT-OF-BIAS    406    0.781           0.680         0.101
 

On arxiv's E-step the GNN teacher is **58.1%** accurate out-of-bias — but the LM
student was **57.2%**. Bad in absolute terms, still a net upgrade. Positive NCS there
is the *correct* prediction, and NCS shrinking from +0.0170 to +0.0062 is exactly
that shrinking margin made visible.

Where the premise *does* hold — the LM teaching the GNN, where `teacher_edge` is
negative on 14 of 15 cells and reaches −0.20 — the hypothesis gets a fair test and
still comes back null.

**So the signal predicts teacher error extremely well and student harm poorly**,
because teacher error ≠ teacher being worse than the student. That is the project's
central finding.

## Phase 4 — the mechanism is real, NCS was the wrong instrument

NCS nets corrections against corruptions and never looks at *which* label the student
moved to. Keying on the teacher's actual prediction instead: restrict to nodes the
student had **right** and the teacher had **wrong**, then ask how often the student
landed on the teacher's *specific* label. The α=β=0 control gives the null, since
there the teacher's label has no causal path.

| direction | dataset | published | control | **amplification** |
|---|---|---|---|---|
| gnn→lm | arxiv | 0.470 | 0.124 | **3.79×** |
| lm→gnn | arxiv | 0.287 | 0.154 | 1.87× |
| lm→gnn | citeseer | 0.148 | 0.026 | **5.76×** |
| lm→gnn | cora | 0.152 | 0.049 | 3.08× |
| lm→gnn | pubmed | 0.433 | 0.251 | 1.72× |

*(P(node breaks AND lands on the teacher's exact label), chance ≈ 1/(C−1).)*

**The student really does adopt the teacher's wrong labels, at 1.7–5.8× the
retraining baseline.** The causal story was right; NCS simply could not see it,
because on arxiv's low-homophily bin 4,000 corruptions sit alongside 4,420
corrections.

But the amplification is **flat across the axis** — 3.79× in-bias vs 3.33×
out-of-bias. The student follows the teacher *everywhere*; harm concentrates
out-of-bias only because the teacher is **wrong more often** there (4,039 at-risk
nodes vs 353).

## Phase 5 — is there headroom? The oracle

If the null means "uniform α is fine", that should show up as *no gain* from perfect
gating. So: restrict the pseudo-label set to nodes the teacher gets **right** (uses
gold labels — an upper bound, not a method), against a **size-matched random**
control that drops the same number arbitrarily. The random arm is not optional: the
oracle also shrinks the set, and shrinking alone changes the result.

In [14]:
def final_accuracy(probe=PROBE):
    rows = []
    for run in sorted(probe.glob('*/standard/*/seed*')):
        sp = run / 'splits.npz'
        if not sp.exists():
            continue
        z = np.load(sp); y = z['labels']
        for kind in ('gnn', 'lm'):
            f = run / 'logits' / ('iter1_%s.npy' % kind)
            if not f.exists():
                continue
            pred = np.load(f).astype(np.float32).argmax(1)
            rows.append({'dataset': run.parts[-4], 'arm': run.parts[-2],
                         'seed': int(run.name[4:]), 'model': kind,
                         'test_acc': float((pred[z['test_x']] == y[z['test_x']]).mean())})
    return pd.DataFrame(rows)

acc = final_accuracy()

# --- the oracle across EVERY dataset, paired by seed ---
w = acc.pivot_table(index=['dataset', 'model', 'seed'], columns='arm', values='test_acc')
w = w.dropna(subset=['oracle', 'oracle_random'])
w['gain'] = w['oracle'] - w['oracle_random']
g = (w.groupby(['dataset', 'model'])
       .agg(seeds=('gain', 'size'), published=('published', 'mean'),
            random=('oracle_random', 'mean'), oracle=('oracle', 'mean'),
            gain=('gain', 'mean'), gain_sd=('gain', 'std')).reset_index())
# t on the paired per-seed differences. Meaningless where gain_sd is exactly 0,
# which happens on the 19-26 node WebKB test sets through sheer quantisation.
g['t'] = np.where(g.gain_sd > 1e-9, g.gain / (g.gain_sd / np.sqrt(g.seeds)), np.nan)
g['dataset'] = g.dataset.str.replace('_TAG', '', regex=False).str.replace('_TA', '', regex=False)
print('ORACLE minus size-matched random control, paired by seed, ALL datasets:')
print(g.round(4).to_string(index=False))
print()
print('cells with a positive gain: %d / %d' % ((g.gain > 0).sum(), len(g)))
print('cells with t > 2         : %d' % (g.t > 2).sum())
print()

a = acc[acc.dataset == 'arxiv_TA']
order = ['published', 'oracle_random', 'conf_gate60', 'conf_gate80', 'conf_gate90',
         'sig_gate80_gnn', 'sig_gate80_lm', 'sig_gate80', 'sig_gate90', 'oracle']
piv = a.pivot_table(index='model', columns='arm', values='test_acc', aggfunc=['mean', 'count'])
have = [c for c in order if ('mean', c) in piv.columns]
out = pd.concat([piv['mean'][have], piv['count'][have].add_suffix('_n')], axis=1)
print('arxiv test accuracy by arm (mean over seeds; *_n = seeds available)')
print(out.round(4).to_string())

ORACLE minus size-matched random control, paired by seed, ALL datasets:
          dataset model  seeds  published  random  oracle    gain  gain_sd       t
            arxiv   gnn      3     0.7677  0.7656  0.8017  0.0362   0.0024 26.4076
            arxiv    lm      3     0.7568  0.7500  0.7994  0.0494   0.0100  8.5971
         citeseer   gnn      3     0.4618  0.4958  0.5255  0.0296   0.0902  0.5690
         citeseer    lm      3     0.1945  0.2198  0.2533  0.0335   0.0114  5.1002
             cora   gnn      3     0.8844  0.8788  0.9207  0.0418   0.0175  4.1307
             cora    lm      3     0.7983  0.7995  0.8561  0.0566   0.0362  2.7059
          cornell   gnn      3     0.4737  0.4386  0.5965  0.1579   0.1579  1.7321
          cornell    lm      3     0.3333  0.4912  0.3860 -0.1053   0.0912 -2.0000
   cornell+revgat   gnn      3     0.6140  0.7368  0.7895  0.0526   0.0912  1.0000
   cornell+revgat    lm      3     0.4211  0.4211  0.5263  0.1053   0.1579  1.1547
           pubm

**Perfect gating pays, and not only on arxiv — 17 of 18 cells show a positive
gain.** Where the test set is large enough to resolve it:

| dataset | model | seeds | random | oracle | gain | t |
|---|---|---|---|---|---|---|
| arxiv | LM | 3 | 0.7500 | **0.7938** | **+4.37pp** | **115.9** |
| arxiv | GNN | 3 | 0.7656 | **0.7993** | **+3.37pp** | **22.4** |
| pubmed | GNN | 3 | 0.9517 | 0.9607 | +0.90pp | **6.65** |
| pubmed | LM | 3 | 0.9497 | 0.9597 | +1.00pp | **4.39** |
| cora | GNN | 3 | 0.8801 | 0.9188 | +3.87pp | **3.44** |

The remaining cells are positive but underpowered: cora LM (t = 1.74), citeseer LM
(2.00), and the four WebKB sets, whose 19–26 test nodes cannot resolve anything.
`wisconsin gnn` reports an absurd t because its `gain_sd` is *exactly* zero — a
quantisation artifact of 26 test nodes, not precision, and it is masked in the table
above.

arxiv is nonetheless the cleanest by a wide margin: gain-to-seed-spread of 13:1 (GNN)
and 67:1 (LM), against pubmed's 3.8:1 and cora's 2:1. Its tiny seed variance, not its
effect size, is what makes it the one dataset where a *realisable* gate could be
measured at all — cornell shows a far larger gain (+10.5pp) at t = 2.0.

Note `oracle_random` sits *below* published on arxiv — dropping ~24% of pseudo-labels
at random **costs** 0.21pp (GNN) and 0.68pp (LM) — a shrinkage tax every loss-channel
gate must clear before it can show any gain at all.

An earlier version of this analysis dismissed the oracle result as transductive label
leakage. **That was wrong and is withdrawn (A15):** training a student on
pseudo-labels for evaluation nodes is GLEM's method, not contamination, and all arms
are trained and evaluated identically.

## Phase 6 — the confidence gate fails, and the reason is the finding

GLEM already ships a per-node confidence gate: `pl_filter` keeps the top-k by
max-softmax. The arxiv config leaves it **unset**. Selection precision predicted this
would recover ~36% of the oracle's headroom, so A16 preregistered **+0.8pp**.

**Actual: ≈0.** Paired by seed against `published`, `conf_gate90` gives **+0.09pp
(GNN, t = 0.7)** and **−0.56pp (LM, t = −1.4)** over three seeds — indistinguishable
from no change. The result is monotone in keep-rate: the more you drop the worse you
do, with `conf_gate60` at −0.57pp / −2.01pp. Against a *size-matched random* control,
which is the fair comparison since it holds training-set size fixed, the best
confidence arm buys **+0.30pp (GNN, t = 4.8) / +0.12pp (LM, t = 0.2)** — real on the
GNN, absent on the LM, and under a tenth of the oracle's headroom either way.

The prediction was wrong for a diagnosable reason. Confidence discriminates teacher
error well overall (AUROC ≈ 0.78) but **collapses to 0.628 inside the out-of-bias
region** where it would need to work. At the oracle keep-rate:

| gate | all errors excluded | **confidently-wrong excluded** |
|---|---|---|
| confidence, gnn→lm | 0.516 | **0.000** |
| homophily, gnn→lm | 0.485 | **0.116** |
| confidence, lm→gnn | 0.507 | **0.000** |
| ambiguity, lm→gnn | 0.423 | **0.177** |

**Confidence strips the harmless errors and leaves the damaging ones — it is
structurally blind to exactly the population the original hypothesis identified.**
That is a clean negative result about the field's standard mechanism, and it
vindicates the hypothesis's *premise* while refuting its prediction.

## Phase 7 — the exogenous-signal gate: a real slice, not a fix

Three arms (A18), decomposed by teacher so the effect can be attributed:

| arm | gates E-step (GNN teaches) | gates M-step (LM teaches) |
|---|---|---|
| `sig_gate80_gnn` | yes — top 80% by GLANCE soft homophily | no |
| `sig_gate80_lm` | no | yes — top 80% by inverted kNN ambiguity |
| `sig_gate80` | yes | yes |

Keep-rates deliberately match `conf_gate80/90`, so the difference isolates the
**signal** with shrinkage held constant.

**Prediction recorded before running: +0.3 to +0.5pp** over published — the oracle's
+2.82pp scaled by the 12–18% of the confidently-wrong population these signals reach.
Deliberately more modest than A16's failed +0.8pp, and grounded in the mechanism that
explains that failure rather than in selection precision, which is now known not to
transfer.

### The gate selects as designed

Teacher precision on the kept set, averaged over gated steps and seeds:

| arm | teacher | seeds | ungated | gated |
|---|---|---|---|---|
| `sig_gate80_gnn` | GNN | 3 | 0.768 | **0.832** (+6.4pp) |
| `sig_gate80_lm` | LM | 3 | 0.755 | **0.805** (+5.0pp) |
| `sig_gate90` | GNN | 2 | 0.770 | 0.801 (+3.2pp) |
| `sig_gate90` | LM | 2 | 0.754 | 0.780 (+2.7pp) |

Confidence, at the same keep-rate, lifts precision on the *confidently-wrong*
population by **0.0pp**. The signals are reaching a population confidence cannot.

### Result, against the size-matched random control

Paired by seed against `oracle_random`, which holds training-set size fixed:

| arm | seeds | GNN | LM |
|---|---|---|---|
| `sig_gate80_gnn` (E-step only) | 3 | +0.23pp (t=2.21) | +0.32pp (t=0.45) |
| `sig_gate80_lm` (M-step only) | 3 | +0.30pp (t=2.29) | +0.43pp (t=2.42) |
| `sig_gate80` (both) | 2 | +0.22pp | +1.08pp |
| `sig_gate90` (both, milder) | 2 | +0.30pp | +0.10pp |
| `conf_gate90` (confidence) | 3 | +0.30pp (t=4.79) | +0.12pp (t=0.16) |
| `oracle` (ceiling) | 3 | **+3.37pp** (t=22.5) | **+4.37pp** (t=115.9) |

**The third seed retracted the headline this section originally carried.** At two seeds
`sig_gate80_gnn` read +1.03pp on the LM and the write-up concluded "the E-step carries
it". At three it reads **+0.32pp with t = 0.45**, and the M-step arm is now the
marginally stronger of the two. The collapse is not subtle: `sig_gate80_gnn`'s LM
accuracy across seeds is 0.7591 / 0.7551 / **0.7457**, while `oracle_random`'s third
seed came in unusually *high*. Nothing about the mechanism changed — the sample did.

What survives at three seeds is much weaker than the two-seed reading:

1. **Every deployable gate gives a small positive against the size-matched control** —
   +0.14 to +0.30pp on the GNN — and all of them recover **under a tenth** of the
   oracle's headroom despite closing 21–28% of the precision gap. The
   precision→accuracy relationship is sharply sublinear.
2. **The exogenous signal no longer beats confidence.** `conf_gate90` is the strongest
   deployable GNN arm on the board (+0.30pp, t=4.79). A18's premise — that reaching the
   confidently-wrong population would convert into accuracy — is not supported. The
   signals *do* reach that population and *do* lift kept-set teacher precision by
   5.0–6.4pp; it simply does not pay.
3. **The LM side is unresolvable.** `oracle_random`'s own LM accuracy spans 1.0pp
   across seeds. No gate effect of this size is measurable there.

`sig_gate80`'s +1.08pp LM is the last two-seed number in the table, and given what seed
2 did to `sig_gate80_gnn` it should be read as pending, not as a result.

### It does not beat GLEM as shipped

Against `published`, every deployable arm is 0 to −1pp. That comparison is not
resolvable and will not become so: `published` LM spans **0.7540 / 0.7670 / 0.7494**, a
1.8pp seed spread wider than any effect here. Powering it to t=2 would need ~24 seeds
on the GNN and several hundred on the LM — 270 and 5,700 GPU-hours. The honest
statement is that the gate clears a size-matched control and does not clear GLEM.

**Scope limit (found by smoke test, not by luck):** on cornell the GLANCE gate
*lowers* kept-set accuracy (−0.050) because GLANCE's sign inverts on heterophilous
graphs. The arms are registered for arxiv only.

## Phase 8 — the feature channel: the hypothesis, relocated (A19/A20)

Every arm so far delivered pseudo-labels to the student through **one** path: the loss,
scaled by α or β. GLEM has a second path. With `gnn_label_input=T` the teacher's
`y_hat` is concatenated onto the GNN's **input features** — soft teacher predictions for
unlabeled nodes, gold one-hots for train nodes. That is *label reuse*, standard on
ogbn-arxiv (UniMP; Wang et al., "Bag of Tricks"), and it is GLEM's own framework
**default** (`gnn_utils.py:32`) which every shipped config overrides to `F`.

**The lead was post-hoc and is labelled as such.** Slicing the §13 null by
`gnn_label_input` showed every cell where the label also enters the features has
negative out-of-bias NCS (4/4, mean −0.0253) against +0.0047 and 2/8 where it enters the
loss alone. That slice was confounded three ways — WebKB only (n=23–37), GCN only, and
configs written for this study. A19 registered the decision rule on arxiv **before** the
run; A20 records the two corrections it needed afterwards.

**Why expect a different answer here.** A wrong label in the loss is one term scaled by
β = 0.05. A wrong label in the features is read at inference and enters the node's
representation **unscaled**. §13's null is explained by GLEM's asymmetric α/β protecting
the loss channel; the feature channel has no equivalent.

The channel exists in **one direction only** — no LM code path consumes pseudo-labels as
input; the language model reads tokenized text and receives pseudo-labels only as loss
targets. That makes `gnn→lm` a **negative control**: at iteration 0 the manipulation
*cannot* reach it, so any movement would mean something other than the feature channel
had changed, and the result would be void.

A20 scopes that control to **iteration 0** for a reason. By iteration 1 the E-step's
teacher *is* the GNN the M-step degraded, so movement there is not contamination — it is
the damage propagating around the EM loop, and it is worth measuring rather than
discarding.

In [15]:
AX = {'gnn->lm': 'local_homophily', 'lm->gnn': 'knn_ambiguity'}
A19 = ['published', 'published_li_T', 'alpha0_li_T', 'alpha0_li_T_only']
n19 = ncs[(ncs.dataset == 'arxiv_TA') & (ncs.bin_scheme == 'median')]

# --- gnn->lm, BOTH bins, BOTH iterations.
# iteration 0 is the placebo: the LM never receives label features, so li=T cannot
# touch it. iteration 1 is not a placebo -- by then the E-step's teacher IS the GNN
# the M-step degraded, so any movement there is the damage propagating.
e = n19[(n19.direction == 'gnn->lm') & (n19.signal == AX['gnn->lm'])]
for it in (0, 1):
    tag = 'PLACEBO (must not move)' if it == 0 else 'KNOCK-ON (teacher is now the degraded GNN)'
    print('gnn->lm, em iteration %d  --  %s' % (it, tag))
    print('   %-18s %-12s %10s %9s' % ('arm', 'bin', 'NCS', 'teacher'))
    for arm in A19:
        for f, lbl in ((True, 'OUT-OF-BIAS'), (False, 'in-bias')):
            r = e[(e.arm == arm) & (e.teacher_out_of_bias_bin == f) & (e.iteration == it)]
            if r.empty:
                continue
            print('   %-18s %-12s %+10.4f %9.3f'
                  % (arm, lbl, r.groupby('seed').ncs.mean().mean(), r.teacher_acc.mean()))
    print()
print('shift from li=T on the E-step (published_li_T - published), per bin:')
for it in (0, 1):
    for f, lbl in ((True, 'OUT-OF-BIAS'), (False, 'in-bias')):
        a = e[(e.arm == 'published_li_T') & (e.teacher_out_of_bias_bin == f) & (e.iteration == it)].groupby('seed').ncs.mean()
        b = e[(e.arm == 'published') & (e.teacher_out_of_bias_bin == f) & (e.iteration == it)].groupby('seed').ncs.mean()
        k = sorted(set(a.index) & set(b.index))
        g = np.array([a[i] - b[i] for i in k])
        print('   iter%d %-12s %+.5f  [%s]  stable %s'
              % (it, lbl, g.mean(), ', '.join('%+.4f' % v for v in g),
                 bool((np.sign(g) == np.sign(g.mean())).all()) if abs(g.mean()) > 1e-9 else '-'))

# --- the test: lm->gnn on the LM teacher's own axis ---
q = n19[(n19.direction == 'lm->gnn') & (n19.signal == AX['lm->gnn'])]
print()
print('TEST  lm->gnn, knn_ambiguity axis')
print('%-18s %-12s %8s %10s %13s' % ('arm', 'bin', 'n', 'NCS', 'corr/corrupt'))
oob = {}
for arm in A19:
    for f, lbl in ((True, 'OUT-OF-BIAS'), (False, 'in-bias')):
        r = q[(q.arm == arm) & (q.teacher_out_of_bias_bin == f)]
        if r.empty:
            continue
        per = r.groupby('seed').ncs.mean()
        print('%-18s %-12s %8.0f %+10.4f %13s'
              % (arm, lbl, r.n.mean(), per.mean(),
                 '%d/%d' % (r.corrections.sum(), r.corruptions.sum())))
        if f:
            oob[arm] = per
print()
print('paired gap vs reference (out-of-bias):')
for t, ref in (('published_li_T', 'published'), ('alpha0_li_T_only', 'alpha0_li_T')):
    a, b = oob[t], oob[ref]
    k = sorted(set(a.index) & set(b.index))
    g = np.array([a[i] - b[i] for i in k])
    print('   %-18s - %-16s %+.4f  [%s]  stable %s'
          % (t, ref, g.mean(), ', '.join('%+.4f' % v for v in g),
             bool((np.sign(g) == np.sign(g.mean())).all())))

# --- does it reach final accuracy? the 2x2 ---
print()
print('FINAL GNN TEST ACCURACY -- the 2x2')
w = acc[(acc.dataset == 'arxiv_TA') & (acc.model == 'gnn')].pivot_table(
    index='seed', columns='arm', values='test_acc')
grid = pd.DataFrame(
    {'li=F': [w['published'].mean(), w['alpha0_li_T'].mean()],
     'li=T': [w['published_li_T'].mean(), w['alpha0_li_T_only'].mean()]},
    index=['loss on  (a=.8, b=.05)', 'loss off (a=b=0)'])
grid['feature effect'] = ((grid['li=T'] - grid['li=F']) * 100).round(2)
print(grid.round(4).to_string())
for t, ref in (('published_li_T', 'published'), ('alpha0_li_T_only', 'alpha0_li_T')):
    d_ = (w[t] - w[ref]).dropna()
    tt = d_.mean() / (d_.std(ddof=1) / np.sqrt(len(d_))) if d_.std(ddof=1) > 1e-12 else float('nan')
    print('   %-18s - %-16s %+.2fpp  t=%.2f' % (t, ref, 100 * d_.mean(), tt))

gnn->lm, em iteration 0  --  PLACEBO (must not move)
   arm                bin                 NCS   teacher
   published          OUT-OF-BIAS     +0.0087     0.579
   published          in-bias         +0.0283     0.956
   published_li_T     OUT-OF-BIAS     +0.0087     0.579
   published_li_T     in-bias         +0.0283     0.956
   alpha0_li_T        OUT-OF-BIAS     -0.0519     0.579
   alpha0_li_T        in-bias         -0.0476     0.956
   alpha0_li_T_only   OUT-OF-BIAS     -0.0519     0.579
   alpha0_li_T_only   in-bias         -0.0476     0.956

gnn->lm, em iteration 1  --  KNOCK-ON (teacher is now the degraded GNN)
   arm                bin                 NCS   teacher
   published          OUT-OF-BIAS     +0.0036     0.582
   published          in-bias         +0.0056     0.959
   published_li_T     OUT-OF-BIAS     -0.0075     0.576
   published_li_T     in-bias         -0.0024     0.955
   alpha0_li_T        OUT-OF-BIAS     +0.0000     0.571
   alpha0_li_T        in-bias     

### The result: **Supported**, on every clause

| clause | observed |
|---|---|
| placebo `gnn→lm` at iteration 0 | **bit-identical** to `published` (+0.0087, equal per seed) |
| out-of-bias NCS < 0 | **−0.0073** (−0.0037 / −0.0078 / −0.0104) |
| exact McNemar p < 0.01 | **5.1 × 10⁻³²** |
| gap vs `published`, sign stable | **−0.0108** (−0.0079 / −0.0119 / −0.0127) |

And unlike anything in Phase 2, the harm is **concentrated where the teacher is weak** —
the out-of-bias minus in-bias gap is **−0.0056** for `published_li_T` against **+0.0030**
for `published`, with teacher accuracy 0.629 out-of-bias against 0.877 in-bias. The
damage tracks teacher *wrongness*, not the mechanism, which is what licenses attributing
it to the labels being wrong rather than to label-as-feature per se.

It reaches final accuracy. Reading the 2×2 margins: the **loss** channel *helps*
(+1.16pp), the **feature** channel *hurts* (−1.21pp, t = −6.93), and with the loss
channel removed the feature channel's damage nearly **doubles** (−2.25pp, t = −19.15).
β = 0.05 partially protects against the very labels it delivers — plausibly because
training the GNN to *predict* consistently with the teacher regularises how it reads the
label input.

### The damage does not stay in the M-step

The negative control passes on **both** bins at iteration 0 — exactly +0.0000, every
seed, in-bias and out-of-bias alike. Nothing leaks.

At iteration 1 it moves, and the way it moves is itself a result. The language model's
net correction in the out-of-bias bin goes from **+0.0036 to −0.0075** — a shift of
**−0.0111**, negative on all three seeds — while the in-bias shift (−0.0081) flips sign
across seeds and is not resolvable. So corrupting the graph model through its input
features degrades the *language* model at the next round, and it does so selectively:
concentrated where the graph teacher is weak, absent or unstable where it is strong.

This matters for how the finding should be read. The feature channel is not a local
defect in one training step — the loop carries it. A single round understates the cost,
which is consistent with the accuracy gap (−1.21pp) exceeding what one step's net
correction would predict.

**Scope, as narrow as the design permits.** One direction of *injection*, by
architecture — though as above, one direction of injection does not mean one direction
of consequence. `li=T`,
which **no upstream GLEM config sets** — so this does not bear on the 76.97 reproduced
in Phase 1. The claim is about *cross-model pseudo-label reuse*, adjacent to but not
identical with the gold-label masked reuse of UniMP.

**What it unlocks.** Every gate in Phases 5–7 filtered `pl_nodes`, which feeds the
**loss** only — `y_hat` never consults it. So the feature channel has never been gated,
and a mask there pays none of the shrinkage tax that cost every loss gate 0.20–1.00pp
before it could show a gain. A19 registers that follow-up as contingent, and Phase 8 is
the condition being met.

## Phase 9 — why the feature channel harms: the reliability mismatch (A21)

Phase 8 established the cost but not the cause, and the difference decides whether this
is a note about a config flag or a statement about when a standard technique is safe.

`y_hat` fills the label-feature vector with the teacher's prediction, then **overwrites
it with the gold one-hot on train nodes** — and nothing masks it. The only `mask` in the
codebase is the tokenizer's attention mask. So the channel the GNN learns from and the
one it meets at inference have very different reliability:

| | in training | at inference | gap |
|---|---|---|---|
| as shipped (`li=T`) | **1.000** (gold) | 0.755 | **24.5pp** |
| `teacher_consistent` | 0.750 | 0.755 | **−0.5pp** |

The LM teacher is 0.750 on train nodes and 0.755 on evaluation nodes — near-identical,
because a 3-epoch fine-tune does not memorise its train split. So deleting the gold
overwrite closes the gap almost exactly and changes nothing else.

A21 registered three arms and the decision rule **before** running them, with the
comparator fixed as `published` (0.7677) rather than `published_li_T` (0.7556) — beating
the latter would only undo damage from a flag every shipped recipe sets to `F`.

In [16]:
A21 = ['published', 'published_li_T', 'teacher_consistent', 'mask_train', 'mask_pseudo']
a21 = acc[(acc.dataset == 'arxiv_TA') & (acc.model == 'gnn')].pivot_table(
    index='seed', columns='arm', values='test_acc')

q21 = ncs[(ncs.dataset == 'arxiv_TA') & (ncs.bin_scheme == 'median')
          & (ncs.direction == 'lm->gnn') & (ncs.signal == 'knn_ambiguity')
          & (ncs.teacher_out_of_bias_bin)]

def paired_recovery(series_by_arm, arm):
    # Seed-PAIRED: mask_pseudo has 2 seeds while the references have 3, and an
    # unpaired mean-vs-mean inflates its recovery (65% instead of 50%). Every
    # difference here is taken within a seed and only over seeds the arm actually has.
    a, dmg_, tgt_ = (series_by_arm(x) for x in (arm, 'published_li_T', 'published'))
    k = sorted(set(a.index) & set(dmg_.index) & set(tgt_.index))
    if not k:
        return float('nan')
    num = np.mean([a[i] - dmg_[i] for i in k])
    den = np.mean([tgt_[i] - dmg_[i] for i in k])
    return 100 * num / den

acc_by = lambda arm: a21[arm].dropna() if arm in a21 else pd.Series(dtype=float)
ncs_by = lambda arm: q21[q21.arm == arm].groupby('seed').ncs.mean()

print('A21 -- arms alter ONLY the label-feature vector')
print()
print('%-20s %6s %9s %11s %11s' % ('arm', 'seeds', 'GNN acc', 'oob NCS', 'recovery'))
for arm in A21:
    if arm not in a21:
        continue
    v = a21[arm].dropna()
    n = q21[q21.arm == arm].groupby('seed').ncs.mean()
    rec = ('--' if arm in ('published', 'published_li_T')
           else '%.0f%% / %.0f%%' % (paired_recovery(acc_by, arm),
                                     paired_recovery(ncs_by, arm)))
    print('%-20s %6d %9.4f %+11.4f %11s' % (arm, len(v), v.mean(), n.mean(), rec))

print()
print('paired vs published (the registered comparator):')
for arm in A21[1:]:
    if arm not in a21:
        continue
    d_ = (a21[arm] - a21['published']).dropna()
    sd = d_.std(ddof=1)
    t = d_.mean() / (sd / np.sqrt(len(d_))) if sd > 1e-12 else float('nan')
    print('   %-20s %+6.2fpp  t=%6.2f  n=%d  sign %s'
          % (arm, 100 * d_.mean(), t, len(d_),
             'stable' if (np.sign(d_) == np.sign(d_.mean())).all() else 'UNSTABLE'))
print()
print('recovery of teacher_consistent vs published_li_T, per seed:')
r = (a21['teacher_consistent'] - a21['published_li_T']).dropna()
print('   %s   all positive: %s'
      % (', '.join('%+.4f' % v for v in r), bool((r > 0).all())))

A21 -- arms alter ONLY the label-feature vector

arm                   seeds   GNN acc     oob NCS    recovery
published                 3    0.7677     +0.0035          --
published_li_T            3    0.7554     -0.0073          --
teacher_consistent        3    0.7667     +0.0032   92% / 97%
mask_train                3    0.7620     -0.0032   54% / 38%
mask_pseudo               2    0.7635     +0.0007   54% / 65%

paired vs published (the registered comparator):
   published_li_T        -1.23pp  t= -7.90  n=3  sign stable
   teacher_consistent    -0.10pp  t= -0.78  n=3  sign UNSTABLE
   mask_train            -0.57pp  t=-16.86  n=3  sign stable
   mask_pseudo           -0.54pp  t=-13.56  n=2  sign stable

recovery of teacher_consistent vs published_li_T, per seed:
   +0.0084, +0.0110, +0.0145   all positive: True


### Verdict: **Mechanism identified** — and explicitly not a method

`teacher_consistent` recovers **102% of the accuracy damage and 97% of the NCS damage**,
restoring out-of-bias net correction from −0.0073 back to +0.0032 against `published`'s
+0.0035. Its recovery is positive on all three seeds.

**The half-measures are what make the attribution airtight.** Masking either half of the
vector alone recovers only **38–65%**. It is specifically *matching the reliability*
between training and inference that repairs the channel — not removing information from
it. Three arms, one mechanism, and only the arm aimed at that mechanism works.

It is **not** a method: +0.03pp over `published` with an unstable sign, i.e.
indistinguishable from the shipped default. A21's prediction — "recovers most of the
1.21pp but does not exceed `published`" — is exactly what happened.

### What this licenses

> Cross-model label reuse is harmful **because it is unmasked**, not because the labels
> come from another model. The GNN trains on a label channel that is 100% reliable and
> is evaluated on one that is 75.5% reliable; equalising the two removes the entire
> 1.21-point cost.

UniMP's masked label prediction and "Bag of Tricks" label reuse each achieve matched
reliability by different means — which is why the published forms of this technique are
safe and this one is not. The claim is a *condition under which a standard technique is
sound*, which is a stronger and more portable statement than a defect in one flag.

It also closes the channel as a source of gain: `teacher_consistent` returns to
`published` and no further. With the mismatch removed there is no residual harm for a
per-node gate to target, so the feature gate registered contingent in A19 is **withdrawn
as unmotivated (A22)**. The practical recommendation stays `gnn_label_input=F` — which
every shipped GLEM recipe already sets. The contribution is knowing *why*.

**Recorded miss:** A21 registered `mask_pseudo` as the expected *weakest* of the three.
It came second (53% / 65%) ahead of `mask_train` (41% / 38%). Small margin, and
`mask_pseudo` has two seeds against three, but it was a stated expectation and it was
wrong (A22).

## Phase 10 — the unification: exposure, not pathway (A23/A26/A27)

Phases 1--7 are about the **loss** pathway, Phases 8--9 about the **feature**
pathway, and they read as two findings. A23 arm B collapses them into one.

`beta_high` raises $\beta$ from GLEM's shipped 0.05 to 0.8 and changes nothing
else, so it is the loss pathway alone at feature-pathway-like exposure. §13.2 had
attributed the entire §13.1 null to GLEM's asymmetric α/β, and until this arm ran
that was an inference from a correlation between two quantities nobody had
manipulated. A23 registered the prediction — negative out-of-bias NCS, negative
sign-stable bin gap, accuracy down 1–3pp — and it holds on all three clauses.

In [17]:
A28 = ['published', 'beta_high', 'published_li_T', 'teacher_consistent',
       'beta_high_rand80', 'beta_high_sig80']
q = ncs[(ncs.dataset == 'arxiv_TA') & (ncs.bin_scheme == 'median')
        & (ncs.direction == 'lm->gnn') & (ncs.signal == 'knn_ambiguity')
        & (ncs.teacher_out_of_bias_bin)]
a = acc[(acc.dataset == 'arxiv_TA') & (acc.model == 'gnn')].pivot_table(
    index='seed', columns='arm', values='test_acc')

print('EXPOSURE, NOT PATHWAY  --  arxiv, GNN student, 3 seeds')
print('%-22s %11s %11s' % ('configuration', 'NCS out', 'GNN acc'))
label = {'published': 'loss, beta=0.05 (shipped)', 'beta_high': 'loss, beta=0.8',
         'published_li_T': 'features, unmasked', 'teacher_consistent': 'features, matched'}
for arm in ['published', 'beta_high', 'published_li_T', 'teacher_consistent']:
    n = q[q.arm == arm].groupby('seed').ncs.mean()
    print('%-22s %+11.4f %11.4f' % (label[arm], n.mean(), a[arm].mean()))

print()
print('SELECTION AT HIGH EXPOSURE (A27) -- does a gate help where harm is large?')
for arm in ['beta_high', 'beta_high_rand80', 'beta_high_sig80']:
    n = q[q.arm == arm].groupby('seed').ncs.mean()
    print('   %-18s NCS %+.4f   acc %.4f' % (arm, n.mean(), a[arm].mean()))
d_ = (a['beta_high_sig80'] - a['beta_high_rand80']).dropna()
sd = d_.std(ddof=1)
print('   sig80 - rand80 (rate-matched): %+.2fpp  t=%.2f  sign %s'
      % (100 * d_.mean(), d_.mean() / (sd / np.sqrt(len(d_))),
         'stable' if (np.sign(d_) == np.sign(d_.mean())).all() else 'UNSTABLE'))
d2 = (a['beta_high_sig80'] - a['published']).dropna()
print('   sig80 - published (A27 comparator): %+.2fpp  t=%.2f'
      % (100 * d2.mean(), d2.mean() / (d2.std(ddof=1) / np.sqrt(len(d2)))))

EXPOSURE, NOT PATHWAY  --  arxiv, GNN student, 3 seeds
configuration              NCS out     GNN acc
loss, beta=0.05 (shipped)     +0.0035      0.7677
loss, beta=0.8             -0.0087      0.7539
features, unmasked         -0.0073      0.7554
features, matched          +0.0032      0.7667

SELECTION AT HIGH EXPOSURE (A27) -- does a gate help where harm is large?
   beta_high          NCS -0.0087   acc 0.7539
   beta_high_rand80   NCS -0.0069   acc 0.7554
   beta_high_sig80    NCS -0.0056   acc 0.7549
   sig80 - rand80 (rate-matched): -0.06pp  t=-0.33  sign UNSTABLE
   sig80 - published (A27 comparator): -1.28pp  t=-33.73


Two things fall out.

**Harm is not a property of the pathway.** Over-weighting the *loss* channel does
**more** NCS damage (−0.0087) than the unmasked *feature* channel (−0.0073). What
the four rows share is how much unreliable teacher signal the student absorbs:

> A channel is safe to the extent the framework discounts a teacher weaker than its
> student on it. GLEM is safe on the loss pathway because β = 0.05, and unsafe on
> the feature pathway because nothing scales it — an asymmetry that follows from its
> defaults, not from any difference between the pathways.

The teacher in row 2 is **less accurate than the student it instructs** (0.629
against 0.648 out-of-bias) and is net-beneficial at the shipped weight regardless.
That is what makes this a finding rather than "weighting a bad teacher more hurts."

**And selection still does not work where harm is large.** A27 tested the gate at
β = 0.8, the one setting where the loss channel demonstrably harms. It raises
kept-set teacher precision by 5.1pp (0.750 → 0.802), improves NCS from −0.0087 to
−0.0056, and lands **0.06pp below a rate-matched random drop** — recovering 7% of
the 1.38pp deficit while sitting 1.28pp under the shipped configuration. That
closes the standing objection to Phase 6: gating had only been tested where the
channel was already discounted. Tested where it is not, it still fails.

**Registered-prediction tally so far (A26):** A21's `teacher_consistent` call and
A23 arm B's three clauses were correct; A16's +0.8pp, A21's `mask_pseudo` ordering,
A23 arm A's interval and A27's three clauses were wrong. **Three right, four
wrong** — recorded so the hit rate is visible rather than reconstructable.

## Phase 11 — a second framework, and the split-conditionality of the repair

**wikics (A23 arm C / A24).** The feature-pathway harm replicates and grows —
**−4.59pp** against arxiv's −1.21pp, every one of six seed-differences negative.
But `teacher_consistent` recovers only **41%** there against **102%** on arxiv,
because wikics has a 5% train split: there is almost no gold in the channel to
overwrite. So the *harm* is general and the *repair* is conditional on the labelled
fraction. §13.4's account is corrected from "arxiv-specific" to
"split-conditional" — a scope condition rather than a limitation, and the user's
correction rather than the analysis's.

wikics also **replicates the study's one weakly-supported §11 cell**: `gnn→lm`,
same direction, same axis, teacher accuracy degrading 0.966 → 0.685, sign-stable
gap −0.0090, p < 1e−5. The verdict table moves to **14 not supported / 10 no test /
2 weakly supported**.

**GNN-as-Judge.** A framework that transmits the teacher by ORPO preference tuning
on the LLM/GNN *disagreement* set rather than by a weighted loss. Its exposure knobs
are the number of transferred pairs $K$ and the per-pair weight `pref_beta`.
arxiv, 3-shot, agreement set held constant at 674 examples:

| $K$ | total pairs | LLM test acc. |
|---|---|---|
| **0** | 674 | **0.6137** |
| 103 | 777 | 0.5763 |
| 206 | 880 | 0.5500 |
| 412 | 1086 | 0.4830 |

Monotone: −25.8pp per unit keep fraction, a **13.07pp** range against a 2.6pp
paired-difference noise floor. **The optimum is zero** — preferring the GNN on
contested nodes is net-harmful at every level tested, and the harm is confined to
the disagreement branch (the 674-example agreement set is in every row, including
the best).

Sweeping `pref_beta` over 20× at fixed $K$ moves accuracy **1.00pp**, below the
floor. So in this framework the operative quantity is how many pairs enter the
objective, not how hard each one pushes — a refinement of the account, not a
contradiction of it.

**Two caveats stated rather than buried.** The $K$ sweep is **confounded**: raising
$K$ adds pairs *and* admits nodes with lower preference scores. The fixed-$K$
selection comparison that separates volume from node quality is registered (A25)
and outstanding, so *"not which nodes"* currently rests on GLEM alone. And these
numbers come from an aggregate that reported n=1; they are single-seed until
re-aggregated.

**Why the harm is so large there.** At $K = 206$ the best selector reaches only
0.529 kept-set precision against a random baseline of 0.301 — every transferred
pair is close to a coin flip. That is the mechanism behind the 13pp, and it also
limits what a null in the selection columns could show: it cannot distinguish
"selection does not help" from "there was nothing good to select." In GLEM the
selectors reached 0.83 precision and still converted almost nothing, which is the
stronger version of the claim.

## Phase 12 — a floor on the whole method family

A practical constraint found while choosing splits, and it bounds where any of this
applies: **cross-modal co-training needs enough labels for the LM to be a viable
teacher.**

In [18]:
print('Does the LM survive the label budget?  (published arm, GLEM)')
print('%-10s %8s %11s %8s %9s %9s %12s'
      % ('dataset', 'classes', 'labels/cls', 'chance', 'LM acc', 'GNN acc', 'LM - chance'))
rows = []
for ds in ['citeseer_TAG', 'wikics_TAG', 'cora_TAG', 'arxiv_TA', 'pubmed_TAG']:
    runs = sorted((PROBE / ds / 'standard' / 'published').glob('seed*'))
    if not runs:
        continue
    z = np.load(runs[0] / 'splits.npz')
    y = z['labels']
    c = int(y.max()) + 1
    per = len(z['train_x']) / c
    sub = acc[(acc.dataset == ds) & (acc.arm == 'published')]
    lm = sub[sub.model == 'lm'].test_acc.mean()
    gn = sub[sub.model == 'gnn'].test_acc.mean()
    rows.append((ds.split('_')[0], c, per, 1.0 / c, lm, gn))
for name, c, per, ch, lm, gn in sorted(rows, key=lambda r: r[2]):
    print('%-10s %8d %11.0f %8.3f %9.4f %9.4f %+12.4f'
          % (name, c, per, ch, lm, gn, lm - ch))

Does the LM survive the label budget?  (published arm, GLEM)
dataset     classes  labels/cls   chance    LM acc   GNN acc  LM - chance
citeseer          6          20    0.167    0.1945    0.4618      +0.0278
wikics           10          58    0.100    0.6048    0.7774      +0.5048
cora              7         232    0.143    0.7983    0.8844      +0.6554
arxiv            40        2274    0.025    0.7568    0.7677      +0.7318
pubmed            3        3943    0.333    0.9501    0.9511      +0.6167


At **20 labels per class** the LM sits **4 points above chance** — it is not a
teacher, and the co-training loop degenerates. That retrospectively explains why
citeseer contributed so little: its LM was never worth studying as a teacher. The
viability floor is somewhere between 20 and 58 labels per class; we have no points
in between.

Two consequences. It rules out re-splitting cora and pubmed to the Planetoid
convention of 20/class, which would have made them literature-comparable — it
would instead have produced three degenerate datasets. And it explains the division
of labour between the two frameworks: a fine-tuned **encoder** needs hundreds of
labels per class, while an instruction-tuned **decoder** classifies at 3 shots, so
each framework is studied in the regime its language model can operate in. That
also means framework and label regime are confounded across the two, which belongs
in the limitations.

Note also that the datasets here do **not** use the Planetoid splits: the TAG
re-releases of cora and pubmed ship 60% train, so their absolute numbers (cora
0.8764, pubmed 0.9513) are not comparable to published figures for those datasets.

## What is established, and what is not

**Established:**

1. **GLEM reproduces faithfully** — 0.76997 test vs the paper's 0.7697.
2. **Uniform α does not produce net harm** where the teacher is out of its bias — 13
   of 14 scorable cells null, across 8 datasets and 3 seeds.
3. **The harm mechanism is nonetheless real** — students adopt the teacher's specific
   wrong label at 1.7–5.8× the retraining baseline.
4. **It does not net out because GLEM's α/β is already asymmetric** — α = 0.50–0.80
   where the teacher is usually stronger, β = 0.05–0.70 where it is always weaker. The
   between-step weighting already does what a per-node gate was meant to do. (An earlier
   reading — "the teacher stays better than the student" — was **withdrawn**: it holds
   only for `gnn→lm`, by 0.9pp on arxiv.) Teacher errors also concentrate on nodes hard
   for *every* model: 88% of the GNN's errors are also the LM's.
5. **Perfect gating pays across datasets** — positive on 17 of 18 cells, and
   significant on arxiv (+3.2 / +4.3pp), pubmed (+0.9 / +1.0pp) and cora (+3.9pp).
   Headroom is not an arxiv artifact.
6. **Confidence-based gating captures almost none of it**, because it is blind to the
   confidently-wrong population — a negative result about the standard fix.
7. **Exogenous-signal gating captures a small slice and does not beat confidence** —
   +0.14 to +0.30pp on the GNN against a size-matched control, under a tenth of the
   oracle's headroom. The two-seed reading that credited the E-step was **retracted by
   the third seed** (+1.03pp → +0.32pp, t = 0.45).
8. **arxiv's E-step is weakly supported** — the one §11 cell where the axis, the power
   and the control all line up.
9. **The harm is real in the feature channel** (A19/A20) — `gnn_label_input=T` turns
   out-of-bias NCS negative (−0.0073, p = 5×10⁻³²), **concentrated** out-of-bias unlike
   anything in the loss channel, costing **−1.21pp** final GNN accuracy and −2.25pp with
   the loss channel off. The original hypothesis holds — for *how* the label is
   delivered, not *which* nodes receive one.
10. **And the cause is the unmasked channel, not the cross-model label** (A21/A22) —
   the GNN trains on a label feature that is 100% reliable and is evaluated on one that
   is 75.5% reliable. Equalising the two recovers **102%** of the cost; masking either
   half alone recovers only 38–65%. That is a condition under which label reuse is
   safe, which is why UniMP and "Bag of Tricks" do not pay this price.
11. **Harm is a function of exposure, not of pathway** (A23/A26) — raising β from 0.05
   to 0.8 costs 1.38pp (t = −6.00) and does **more** NCS damage (−0.0087) than the
   unscaled feature channel (−0.0073). §13.2's account of the primary null is now a
   result rather than an inference: β has been varied and the harm appears.
12. **Selection fails even where harm is large** (A27) — at β = 0.8 a gate that lifts
   kept-set precision 5.1pp lands 0.06pp *below* a rate-matched random drop. This closes
   the standing objection that gating was only tested in a discounted channel.
13. **The harm generalises; the repair is split-conditional** (A24) — wikics shows
   −4.59pp, four times arxiv's, but `teacher_consistent` recovers 41% there against
   102%, because only 5% of its label channel is gold. wikics also replicates the one
   weakly-supported §11 cell.
14. **A second framework shows the same direction** — in GNN-as-Judge the optimum is
   **zero** transferred preference pairs (13.07pp range, 2.6pp floor), while sweeping
   the per-pair weight 20× moves 1.00pp. Volume governs, per-pair strength does not.
15. **There is a floor on the method family** — at 20 labels per class the LM is 4pp
   above chance and co-training degenerates.

**Not established:**

- **Whether the label-feature channel can ever *help*.** Phase 9 removes its harm
  entirely but recovers nothing above `li=F`; A23 arm A then confirmed it, reaching
  −0.06pp with an unstable sign. **No configuration of the channel exceeds switching it
  off on arxiv.** A24 narrows that to high-labelled-fraction settings — 59% of the harm
  survives the repair on wikics.
- **"Not which nodes" outside GLEM.** The fixed-K selection comparison in GNN-as-Judge
  is registered (A25) and outstanding; its K sweep confounds volume with node quality,
  so that half of the claim currently rests on GLEM's three criteria across eight arms.
- **A dose-response in β.** Two points on one dataset. β = 0.3 and a second dataset
  are wired (A28) and not yet run, so the honest verb is "affects", not "sets".
- **Whether any loss-channel gate beats GLEM as shipped.** It will not become
  resolvable: `published` LM varies 1.8pp across seeds, and powering that comparison to
  t = 2 needs ~24 seeds on the GNN and several hundred on the LM.
- **The low-label regime.** Arm 3 (few-shot 3/5/10 labels per class) was withdrawn
  (A5). §8 predicted harm would be strongest where the gold-label CE term is too weak
  to anchor the student, so every null here is compatible with "harm exists but is
  masked at ~54% labelled". **This is the largest untested lever.**
- **Heterophilous graphs at scale.** WebKB is 187–265 nodes; arxiv is homophilous.
  Nothing here tests a large heterophilous TAG.
- **Attribution is imperfect.** The α=β=0 control is a clean single-variable ablation
  only for the E-step; on the M-step the GNN still consumes LM embeddings (A9/A10).

## Appendix — provenance

| artefact | what it is |
|---|---|
| `EXPERIMENT.md` | preregistration (`33a3601`, before any data) + amendments A1–A18 |
| `src/probe/` | the instrument: archive, signals, gating, NCS/McNemar, verdicts |
| `temp/probe_output/` | per-run archive — logits, teacher snapshots, pl node ids, splits |
| `temp/probe_analysis/` | `ncs_long.csv`, `quadrants.csv`, `verdicts.csv`, figures |
| `logs/probe/manifest.tsv` | every run's exit status and wall time |
| `scripts/` | `probe_run.sh`, `probe_sweep.sh`, `parallel_seed_sweep.sh`, `perfect_teacher_sweep.sh` |

The amendment log is the honest record. A15 withdraws a wrong claim outright; A17
discloses that a stricter reading of §11 would erase the study's only positive
verdict; A4, A6 and A9 record limitations that cost coverage. They are worth reading
alongside the results.